# D4BL Training Pipeline — Qwen 3.5-4B

Run the headless training script on Colab A100. Trains 3 LoRA adapters (query parser, explainer, evaluator) with domain adaptation, then exports to GGUF. All outputs are backed up to Google Drive.

**Runtime:** Change to **A100** GPU via Runtime > Change runtime type.

**Time estimate:** ~30-50 minutes on A100.

## 0. Mount Google Drive

Mounts Drive for backup/restore. If a previous training run exists on Drive, it will be restored so the script can resume.

In [ ]:
from google.colab import drive
import shutil, os

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/d4bl_training"
LOCAL_DIR = "/content/d4bl_training"

# Restore from Drive if a previous run exists
if os.path.exists(DRIVE_DIR):
    print(f"Found existing training state on Drive: {DRIVE_DIR}")
    shutil.copytree(DRIVE_DIR, LOCAL_DIR, dirs_exist_ok=True)
    items = sum(1 for _, _, files in os.walk(LOCAL_DIR) for _ in files)
    print(f"Restored {items} files — training script will resume from checkpoints")
else:
    print("No previous training state on Drive — starting fresh")

## 1. Verify GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Install dependencies

In [ ]:
import subprocess, sys

# Unsloth (includes transformers, peft, trl, etc.)
subprocess.run([sys.executable, "-m", "pip", "install", "unsloth"], check=True)

# HuggingFace datasets
subprocess.run([sys.executable, "-m", "pip", "install", "datasets"], check=True)

## 3. Clone repo and set up

In [ ]:
import os
os.chdir("/content")

# Clone if not already present
if not os.path.exists("/content/d4bl_ai_agent"):
    !git clone https://github.com/William-Hill/d4bl_ai_agent.git

os.chdir("/content/d4bl_ai_agent")
!git pull origin main
!pwd

## 4. (Optional) HuggingFace token for gated models

In [ ]:
# Uncomment and set if needed for gated model access
# import os
# os.environ["HF_TOKEN"] = "hf_your_token_here"

## 5. Run training

This runs all 5 phases: domain adaptation, 3 task adapters, and GGUF export. Progress streams in real-time. If interrupted, rerun this cell — it resumes from checkpoints.

In [ ]:
!python scripts/training/train.py \
    --data-dir scripts/training_data/final \
    --output-dir /content/d4bl_training

In [ ]:
import shutil

DRIVE_DIR = "/content/drive/MyDrive/d4bl_training"
LOCAL_DIR = "/content/d4bl_training"

print("Backing up training outputs to Google Drive...")
shutil.copytree(LOCAL_DIR, DRIVE_DIR, dirs_exist_ok=True)
items = sum(1 for _, _, files in os.walk(DRIVE_DIR) for _ in files)
print(f"Backed up {items} files to {DRIVE_DIR}")

## 7. Review results

In [ ]:
# Training report
with open("/content/d4bl_training/training_report.md") as f:
    from IPython.display import Markdown, display
    display(Markdown(f.read()))

In [ ]:
# List GGUF exports
!ls -lh /content/d4bl_training/gguf/*/

## 8. Download GGUFs

Download the GGUF files to use locally with Ollama. (Also available on Drive at `MyDrive/d4bl_training/gguf/`)

In [ ]:
from google.colab import files
import glob

for gguf in glob.glob("/content/d4bl_training/gguf/**/*.gguf", recursive=True):
    print(f"Downloading {gguf}...")
    files.download(gguf)